In [33]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from geopy.distance import geodesic
from itertools import combinations
import folium
from folium.plugins import HeatMap
from scipy.stats import pearsonr
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")

Libraries imported successfully!


# Sensor Distance Analysis and Coverage Estimation

This notebook analyzes sensor distances and PM2.5 similarity to estimate the **maximum effective coverage distance** of air quality sensors.

## Objectives:
1. Compute pairwise distances between sensors
2. Analyze PM2.5 similarity vs distance
3. Define maximum coverage distance
4. Visualize coverage zones on map
5. Provide insights for sensor placement optimization

## 1. Load and Prepare Data

In [34]:
# Load the encoded dataset
df = pd.read_csv('data_encoded.csv')
print(f"Dataset shape: {df.shape}")
print("\nColumns:", df.columns.tolist())
print("\nFirst few rows:")
df.head()

Dataset shape: (377629, 14)

Columns: ['timestamp', 'sensor_id', 'sensor_type', 'location', 'lat', 'lon', 'PM10', 'PM2_5', 'humidity', 'temperature', 'date', 'hour', 'day_of_week', 'day_of_month']

First few rows:


,timestamp,sensor_id,sensor_type,location,lat,lon,PM10,PM2_5,humidity,temperature,date,hour,day_of_week,day_of_month
0,2018-09-01 00:00:02.472867+00:00,57,SDS011,29,-1.300,36.785,39.67,34.43,NaN,NaN,2018-09-01,0,Saturday,1
1,2018-09-01 00:00:04.301785+00:00,58,DHT22,29,-1.300,36.785,NaN,NaN,83.4,16.5,2018-09-01,0,Saturday,1
2,2018-09-01 00:00:07.536156+00:00,69,SDS011,7,-1.298,36.791,18.47,16.63,NaN,NaN,2018-09-01,0,Saturday,1
3,2018-09-01 00:00:08.902190+00:00,70,DHT22,7,-1.298,36.791,NaN,NaN,56.5,21.7,2018-09-01,0,Saturday,1
4,2018-09-01 00:00:26.722032+00:00,40,SDS011,7,-1.298,36.791,21.52,19.30,NaN,NaN,2018-09-01,0,Saturday,1


In [35]:
# Get unique sensor locations and their coordinates
sensor_locations = df[df['sensor_type'] == 'SDS011'][['sensor_id', 'location', 'lat', 'lon']].drop_duplicates()
print(f"Number of unique PM2.5 sensors: {len(sensor_locations)}")
print("\nSensor locations:")
print(sensor_locations)

# Check for potential issues
print("\n" + "="*50)
print("DIAGNOSTIC:")
print(f"Unique sensor IDs: {sensor_locations['sensor_id'].nunique()}")
print(f"Total rows: {len(sensor_locations)}")

# Check if any sensor_id appears multiple times with different locations
sensor_id_counts = sensor_locations['sensor_id'].value_counts()
print(f"\nSensor IDs appearing multiple times:")
print(sensor_id_counts[sensor_id_counts > 1])

# Check for duplicate coordinates
coord_counts = sensor_locations.groupby(['lat', 'lon']).size().reset_index(name='count')
print(f"\nDuplicate coordinates:")
print(coord_counts[coord_counts['count'] > 1])

Number of unique PM2.5 sensors: 43

Sensor locations:
        sensor_id  location    lat     lon
0              57        29 -1.300  36.785
2              69         7 -1.298  36.791
4              40         7 -1.298  36.791
5              46        25 -1.301  36.754
8              55        30 -1.290  36.777
11             27         8 -1.306  36.733
14             87         7 -1.298  36.791
16             75         7 -1.298  36.791
18             51        27 -1.306  36.773
22             71         7 -1.298  36.791
24             81        33 -1.255  36.694
25             73        34 -1.206  36.781
3134           89         7 -1.298  36.791
4445           53        28 -1.291  36.725
4448           44        26 -1.320  36.885
10685          71        35 -1.297  36.743
23669          59        31 -1.291  36.781
35030          93         7 -1.298  36.791
35658          97         7 -1.298  36.791
35811          89        36 -1.291  36.733
51132          93        37 -1.316  36.870


## 2. Compute Pairwise Distances Between Sensors

In [36]:
def compute_sensor_distances(sensor_df):
    """
    Compute geodesic distances between all sensor pairs
    """
    # Create unique sensor identifier combining sensor_id and location
    sensor_df = sensor_df.copy()
    sensor_df['unique_sensor_id'] = sensor_df['sensor_id'].astype(str) + '_' + sensor_df['location'].astype(str)
    
    sensors = sensor_df[['unique_sensor_id', 'sensor_id', 'location', 'lat', 'lon']].values
    n_sensors = len(sensors)
    
    distance_matrix = np.zeros((n_sensors, n_sensors))
    sensor_pairs = []
    
    for i, j in combinations(range(n_sensors), 2):
        coord1 = (sensors[i][3], sensors[i][4])  # (lat, lon)
        coord2 = (sensors[j][3], sensors[j][4])
        
        distance = geodesic(coord1, coord2).kilometers
        distance_matrix[i][j] = distance
        distance_matrix[j][i] = distance
        
        sensor_pairs.append({
            'sensor_1': sensors[i][1],  # Original sensor_id
            'sensor_2': sensors[j][1],
            'location_1': sensors[i][2],
            'location_2': sensors[j][2],
            'unique_sensor_1': sensors[i][0],
            'unique_sensor_2': sensors[j][0],
            'distance_km': distance
        })
    
    return distance_matrix, pd.DataFrame(sensor_pairs), sensor_df

# Compute distances
distance_matrix, sensor_pairs_df, unique_sensors_df = compute_sensor_distances(sensor_locations)

print(f"Computed distances for {len(sensor_pairs_df)} sensor pairs")
print(f"Number of unique sensors (by ID+location): {len(unique_sensors_df)}")
print("\nDistance statistics (km):")
print(sensor_pairs_df['distance_km'].describe())

print("\nFirst few sensor pairs:")
print(sensor_pairs_df[['sensor_1', 'location_1', 'sensor_2', 'location_2', 'distance_km']].head())

print("\n" + "="*50)
print("VERIFICATION:")
print(f"Any pairs with sensor_1 == sensor_2: {(sensor_pairs_df['sensor_1'] == sensor_pairs_df['sensor_2']).any()}")
print(f"Any pairs with distance == 0: {(sensor_pairs_df['distance_km'] == 0).any()}")

Computed distances for 903 sensor pairs
Number of unique sensors (by ID+location): 43

Distance statistics (km):
count    903.000000
mean       7.576600
std        5.946835
min        0.000000
25%        2.510053
50%        6.830043
75%       11.752530
max       29.891530
Name: distance_km, dtype: float64

First few sensor pairs:
   sensor_1  location_1  sensor_2  location_2  distance_km
0        57          29        69           7     0.703415
1        57          29        40           7     0.703415
2        57          29        46          25     3.451793
3        57          29        55          30     1.419636
4        57          29        27           8     5.825032

VERIFICATION:
Any pairs with sensor_1 == sensor_2: True
Any pairs with distance == 0: True


## 3. Analyze PM2.5 Similarity vs Distance

In [ ]:
def compute_pm25_differences_instantaneous(df, sensor_pairs_df):
    """
    Compute PM2.5 differences for each timestamp t between all sensor pairs
    Returns a DataFrame with one row per timestamp per sensor pair
    """
    results = []
    
    print(f"Analyzing {len(sensor_pairs_df)} sensor pairs...")
    
    # First, get all PM2.5 data in a convenient format
    pm25_data = df[df['sensor_type'] == 'SDS011'][['timestamp', 'sensor_id', 'location', 'PM2_5']].copy()
    pm25_data = pm25_data.dropna(subset=['PM2_5'])
    
    if len(pm25_data) == 0:
        print("ERROR: No PM2.5 data found!")
        return pd.DataFrame()
    
    print(f"Found {len(pm25_data)} PM2.5 measurements")
    print(f"PM2.5 column name: 'PM2_5'")
    print(f"Sample PM2.5 values: {pm25_data['PM2_5'].head().tolist()}")
    
    # Convert timestamp to datetime with error handling
    try:
        pm25_data['timestamp'] = pd.to_datetime(pm25_data['timestamp'], format='ISO8601')
    except:
        try:
            pm25_data['timestamp'] = pd.to_datetime(pm25_data['timestamp'])
        except Exception as e:
            print(f"Error converting timestamps: {e}")
            return pd.DataFrame()
    
    # Round timestamps to the nearest minute to find common measurement points
    pm25_data['rounded_timestamp'] = pm25_data['timestamp'].dt.round('min')
    
    # Group by rounded_timestamp, sensor_id, location and take mean PM2.5 if multiple readings in same minute
    pm25_data_agg = pm25_data.groupby(['rounded_timestamp', 'sensor_id', 'location'])['PM2_5'].mean().reset_index()
    pm25_data_agg = pm25_data_agg.rename(columns={'rounded_timestamp': 'timestamp'})
    
    print(f"Found {len(pm25_data_agg['timestamp'].unique())} unique minute-level timestamps after aggregation")
    
    # Check how many sensors have data at each timestamp
    timestamp_counts = pm25_data_agg.groupby('timestamp').size().reset_index(name='sensor_count')
    multi_sensor_timestamps = timestamp_counts[timestamp_counts['sensor_count'] >= 2]
    
    print(f"Found {len(multi_sensor_timestamps)} timestamps with 2+ sensors reporting data")
    
    if len(multi_sensor_timestamps) == 0:
        print("WARNING: No timestamps found with multiple sensors!")
        print("Trying with 5-minute intervals...")
        
        # Try with 5-minute intervals
        pm25_data['rounded_timestamp'] = pm25_data['timestamp'].dt.round('5min')
        pm25_data_agg = pm25_data.groupby(['rounded_timestamp', 'sensor_id', 'location'])['PM2_5'].mean().reset_index()
        pm25_data_agg = pm25_data_agg.rename(columns={'rounded_timestamp': 'timestamp'})
        
        timestamp_counts = pm25_data_agg.groupby('timestamp').size().reset_index(name='sensor_count')
        multi_sensor_timestamps = timestamp_counts[timestamp_counts['sensor_count'] >= 2]
        
        print(f"With 5-minute intervals: {len(multi_sensor_timestamps)} timestamps with 2+ sensors")
    
    # Create a pivot table for easier comparison
    pivot_data = pm25_data_agg.pivot_table(
        index='timestamp', 
        columns=['sensor_id', 'location'], 
        values='PM2_5'
    )
    
    print(f"Pivot table shape: {pivot_data.shape}")
    print(f"Sensors in pivot: {list(pivot_data.columns)}")
    
    # Get all sensor pairs that have data
    available_sensors = [(sid, loc) for sid, loc in pivot_data.columns]
    
    pair_count = 0
    for i, (sensor_1, location_1) in enumerate(available_sensors):
        for j, (sensor_2, location_2) in enumerate(available_sensors):
            if i >= j:  # Skip same sensor and avoid duplicates
                continue
                
            # Find distance for this pair
            pair_info = sensor_pairs_df[
                ((sensor_pairs_df['sensor_1'] == sensor_1) & (sensor_pairs_df['location_1'] == location_1) &
                 (sensor_pairs_df['sensor_2'] == sensor_2) & (sensor_pairs_df['location_2'] == location_2)) |
                ((sensor_pairs_df['sensor_1'] == sensor_2) & (sensor_pairs_df['location_1'] == location_2) &
                 (sensor_pairs_df['sensor_2'] == sensor_1) & (sensor_pairs_df['location_2'] == location_1))
            ]
            
            if len(pair_info) == 0:
                continue
                
            distance = pair_info['distance_km'].iloc[0]
            
            # Get common timestamps where both sensors have data
            common_data = pivot_data[[ (sensor_1, location_1), (sensor_2, location_2) ]].dropna()
            
            if len(common_data) == 0:
                continue
                
            pair_count += 1
            print(f"  Pair {sensor_1}-{location_1} vs {sensor_2}-{location_2}: {len(common_data)} common timestamps, distance: {distance:.2f}km")
            
            # For each common timestamp, compute PM2.5 difference
            for timestamp, row in common_data.iterrows():
                pm25_1 = row[(sensor_1, location_1)]
                pm25_2 = row[(sensor_2, location_2)]
                pm25_diff = abs(pm25_1 - pm25_2)
                
                results.append({
                    'timestamp': timestamp,
                    'sensor_1': sensor_1,
                    'sensor_2': sensor_2,
                    'location_1': location_1,
                    'location_2': location_2,
                    'distance_km': distance,
                    'pm25_1': pm25_1,
                    'pm25_2': pm25_2,
                    'pm25_diff': pm25_diff
                })
    
    print(f"Processed {pair_count} valid sensor pairs")
    
    if len(results) == 0:
        print("WARNING: No common timestamps found between any sensor pairs!")
        return pd.DataFrame()
    
    result_df = pd.DataFrame(results)
    print(f"Generated {len(result_df)} instantaneous PM2.5 difference records")
    
    return result_df

# Compute instantaneous PM2.5 differences
pm25_instantaneous_df = compute_pm25_differences_instantaneous(df, sensor_pairs_df)

if len(pm25_instantaneous_df) > 0:
    print(f"\nInstantaneous PM2.5 Difference Statistics:")
    print(pm25_instantaneous_df['pm25_diff'].describe())
    
    print(f"\nDistance Statistics:")
    print(pm25_instantaneous_df['distance_km'].describe())
    
    print(f"\nFirst few records:")
    print(pm25_instantaneous_df[['timestamp', 'sensor_1', 'sensor_2', 'distance_km', 'pm25_diff']].head())
else:
    print("ERROR: No instantaneous PM2.5 data generated!")

,timestamp,sensor_id,sensor_type,location,lat,lon,PM10,PM2_5,humidity,temperature,date,hour,day_of_week,day_of_month
0,2018-09-01 00:00:02.472867+00:00,57,SDS011,29,-1.300,36.785,39.67,34.43,NaN,NaN,2018-09-01,0,Saturday,1
1,2018-09-01 00:00:04.301785+00:00,58,DHT22,29,-1.300,36.785,NaN,NaN,83.4,16.5,2018-09-01,0,Saturday,1
2,2018-09-01 00:00:07.536156+00:00,69,SDS011,7,-1.298,36.791,18.47,16.63,NaN,NaN,2018-09-01,0,Saturday,1
3,2018-09-01 00:00:08.902190+00:00,70,DHT22,7,-1.298,36.791,NaN,NaN,56.5,21.7,2018-09-01,0,Saturday,1
4,2018-09-01 00:00:26.722032+00:00,40,SDS011,7,-1.298,36.791,21.52,19.30,NaN,NaN,2018-09-01,0,Saturday,1


In [ ]:
def compute_pm25_differences_instantaneous(df, sensor_pairs_df):
    """
    Compute PM2.5 differences for each timestamp t between all sensor pairs
    Returns a DataFrame with one row per timestamp per sensor pair
    """
    results = []w
    
    print(f"Analyzing {len(sensor_pairs_df)} sensor pairs...")
    
    # First, get all PM2.5 data in a convenient format
    pm25_data = df[df['sensor_type'] == 'SDS011'][['timestamp', 'sensor_id', 'location', 'PM2_5']].copy()
    pm25_data = pm25_data.dropna(subset=['PM2_5'])
    
    if len(pm25_data) == 0:
        print("ERROR: No PM2.5 data found!")
        return pd.DataFrame()
    
    print(f"Found {len(pm25_data)} PM2.5 measurements")
    
    # Convert timestamp to datetime if it's not already
    pm25_data['timestamp'] = pd.to_datetime(pm25_data['timestamp'])
    
    # Round timestamps to the nearest minute to find common measurement points
    pm25_data['rounded_timestamp'] = pm25_data['timestamp'].dt.round('min')
    
    # Group by rounded_timestamp, sensor_id, location and take mean PM2.5 if multiple readings in same minute
    pm25_data_agg = pm25_data.groupby(['rounded_timestamp', 'sensor_id', 'location'])['PM2_5'].mean().reset_index()
    pm25_data_agg = pm25_data_agg.rename(columns={'rounded_timestamp': 'timestamp'})
    
    print(f"Found {len(pm25_data_agg['timestamp'].unique())} unique minute-level timestamps after aggregation")
    
    # Check how many sensors have data at each timestamp
    timestamp_counts = pm25_data_agg.groupby('timestamp').size().reset_index(name='sensor_count')
    multi_sensor_timestamps = timestamp_counts[timestamp_counts['sensor_count'] >= 2]
    
    print(f"Found {len(multi_sensor_timestamps)} timestamps with 2+ sensors reporting data")
    
    if len(multi_sensor_timestamps) == 0:
        print("WARNING: No timestamps found with multiple sensors!")
        print("Trying with 5-minute intervals...")
        
        # Try with 5-minute intervals
        pm25_data['rounded_timestamp'] = pm25_data['timestamp'].dt.round('5min')
        pm25_data_agg = pm25_data.groupby(['rounded_timestamp', 'sensor_id', 'location'])['PM2_5'].mean().reset_index()
        pm25_data_agg = pm25_data_agg.rename(columns={'rounded_timestamp': 'timestamp'})
        
        timestamp_counts = pm25_data_agg.groupby('timestamp').size().reset_index(name='sensor_count')
        multi_sensor_timestamps = timestamp_counts[timestamp_counts['sensor_count'] >= 2]
        
        print(f"With 5-minute intervals: {len(multi_sensor_timestamps)} timestamps with 2+ sensors")
    
    for idx, row in sensor_pairs_df.iterrows():
        sensor_1 = row['sensor_1']
        sensor_2 = row['sensor_2']
        location_1 = row['location_1']
        location_2 = row['location_2']
        distance = row['distance_km']
        
        # Skip if same sensor
        if sensor_1 == sensor_2 and location_1 == location_2:
            continue
        
        # Get data for both sensors using aggregated data
        data1 = pm25_data_agg[(pm25_data_agg['sensor_id'] == sensor_1) & 
                             (pm25_data_agg['location'] == location_1)]
        data2 = pm25_data_agg[(pm25_data_agg['sensor_id'] == sensor_2) & 
                             (pm25_data_agg['location'] == location_2)]
        
        if len(data1) == 0 or len(data2) == 0:
            continue
        
        # Find common timestamps after rounding
        common_timestamps = set(data1['timestamp']).intersection(set(data2['timestamp']))
        
        if len(common_timestamps) == 0:
            continue
        
        print(f"  Pair {sensor_1}-{location_1} vs {sensor_2}-{location_2}: {len(common_timestamps)} common timestamps")
        
        # For each common timestamp, compute PM2.5 difference
        for timestamp in common_timestamps:
            pm25_1 = data1[data1['timestamp'] == timestamp]['PM2_5'].iloc[0]
            pm25_2 = data2[data2['timestamp'] == timestamp]['PM2_5'].iloc[0]
            
            pm25_diff = abs(pm25_1 - pm25_2)
            
            results.append({
                'timestamp': timestamp,
                'sensor_1': sensor_1,
                'sensor_2': sensor_2,
                'location_1': location_1,
                'location_2': location_2,
                'distance_km': distance,
                'pm25_1': pm25_1,
                'pm25_2': pm25_2,
                'pm25_diff': pm25_diff
            })
    
    if len(results) == 0:
        print("WARNING: No common timestamps found between any sensor pairs!")
        return pd.DataFrame()
    
    result_df = pd.DataFrame(results)
    print(f"Generated {len(result_df)} instantaneous PM2.5 difference records")
    
    return result_df

# Compute instantaneous PM2.5 differences
pm25_instantaneous_df = compute_pm25_differences_instantaneous(df, sensor_pairs_df)

if len(pm25_instantaneous_df) > 0:
    print(f"\nInstantaneous PM2.5 Difference Statistics:")
    print(pm25_instantaneous_df['pm25_diff'].describe())
    
    print(f"\nDistance Statistics:")
    print(pm25_instantaneous_df['distance_km'].describe())
    
    print(f"\nFirst few records:")
    print(pm25_instantaneous_df[['timestamp', 'sensor_1', 'sensor_2', 'distance_km', 'pm25_diff']].head())
else:
    print("ERROR: No instantaneous PM2.5 data generated!")

Analyzing 903 sensor pairs...
Found 218112 PM2.5 measurements


ValueError: time data "2018-09-29 06:11:24+00:00" doesn't match format "%Y-%m-%d %H:%M:%S.%f%z". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

## 4. Visualize PM2.5 Difference vs Distance

In [52]:
# Check if we have instantaneous data before plotting
if 'pm25_instantaneous_df' not in locals() or len(pm25_instantaneous_df) == 0:
    print("ERROR: No instantaneous PM2.5 analysis data available!")
    print("Please run the previous cell first to generate the data.")
elif 'distance_km' not in pm25_instantaneous_df.columns:
    print("ERROR: 'distance_km' column not found in pm25_instantaneous_df")
    print(f"Available columns: {pm25_instantaneous_df.columns.tolist()}")
elif 'pm25_diff' not in pm25_instantaneous_df.columns:
    print("ERROR: 'pm25_diff' column not found in pm25_instantaneous_df")
    print(f"Available columns: {pm25_instantaneous_df.columns.tolist()}")
else:
    # Create comprehensive plots
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))

    # Plot 1: PM2.5 difference vs distance (instantaneous)
    axes[0, 0].scatter(pm25_instantaneous_df['distance_km'], pm25_instantaneous_df['pm25_diff'], 
                       alpha=0.3, s=20)
    axes[0, 0].set_xlabel('Distance (km)')
    axes[0, 0].set_ylabel('PM2.5 Difference (µg/m³)')
    axes[0, 0].set_title('PM2.5 Difference vs Distance (Instantaneous)')
    axes[0, 0].grid(True, alpha=0.3)

    # Add trend line
    z = np.polyfit(pm25_instantaneous_df['distance_km'], pm25_instantaneous_df['pm25_diff'], 1)
    p = np.poly1d(z)
    axes[0, 0].plot(pm25_instantaneous_df['distance_km'], p(pm25_instantaneous_df['distance_km']), "r--", alpha=0.8)

    # Plot 2: Heatmap of PM2.5 difference density
    # Create bins for heatmap
    distance_bins = np.linspace(pm25_instantaneous_df['distance_km'].min(), 
                                pm25_instantaneous_df['distance_km'].max(), 20)
    diff_bins = np.linspace(0, pm25_instantaneous_df['pm25_diff'].max(), 20)
    
    H, xedges, yedges = np.histogram2d(pm25_instantaneous_df['distance_km'], 
                                       pm25_instantaneous_df['pm25_diff'], 
                                       bins=[distance_bins, diff_bins])
    
    im = axes[0, 1].imshow(H.T, origin='lower', aspect='auto', 
                          extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]], 
                          cmap='YlOrRd')
    axes[0, 1].set_xlabel('Distance (km)')
    axes[0, 1].set_ylabel('PM2.5 Difference (µg/m³)')
    axes[0, 1].set_title('Density Heatmap: PM2.5 Diff vs Distance')
    plt.colorbar(im, ax=axes[0, 1])

    # Plot 3: Histogram of distances
    axes[1, 0].hist(pm25_instantaneous_df['distance_km'], bins=20, alpha=0.7, edgecolor='black')
    axes[1, 0].set_xlabel('Distance (km)')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].set_title('Distribution of Distances (All Timestamps)')
    axes[1, 0].grid(True, alpha=0.3)

    # Plot 4: Histogram of PM2.5 differences
    axes[1, 1].hist(pm25_instantaneous_df['pm25_diff'], bins=20, alpha=0.7, 
                    edgecolor='black', color='green')
    axes[1, 1].set_xlabel('PM2.5 Difference (µg/m³)')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].set_title('Distribution of PM2.5 Differences (All Timestamps)')
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Print correlation between distance and PM2.5 difference
    corr_dist_diff = pearsonr(pm25_instantaneous_df['distance_km'], pm25_instantaneous_df['pm25_diff'])[0]
    print(f"Correlation between distance and PM2.5 difference: {corr_dist_diff:.3f}")
    
    # Additional statistics
    print(f"\nTotal instantaneous measurements: {len(pm25_instantaneous_df)}")
    print(f"Number of unique sensor pairs: {pm25_instantaneous_df[['sensor_1', 'sensor_2']].drop_duplicates().shape[0]}")
    print(f"Number of unique timestamps: {pm25_instantaneous_df['timestamp'].nunique()}")

ERROR: No instantaneous PM2.5 analysis data available!
Please run the previous cell first to generate the data.


## 5. Define Maximum Coverage Distance

In [ ]:
# Check if we have instantaneous data before computing coverage thresholds
if 'pm25_instantaneous_df' not in locals() or len(pm25_instantaneous_df) == 0:
    print("ERROR: No instantaneous PM2.5 analysis data available!")
    print("Please run the previous cells first to generate the data.")
elif 'distance_km' not in pm25_instantaneous_df.columns:
    print("ERROR: 'distance_km' column not found in pm25_instantaneous_df")
    print(f"Available columns: {pm25_instantaneous_df.columns.tolist()}")
elif 'pm25_diff' not in pm25_instantaneous_df.columns:
    print("ERROR: 'pm25_diff' column not found in pm25_instantaneous_df")
    print(f"Available columns: {pm25_instantaneous_df.columns.tolist()}")
else:
    def find_coverage_threshold_instantaneous(pm25_instantaneous_df, threshold_diff=5.0):
        """
        Find maximum distance where PM2.5 difference is below threshold
        Based on instantaneous measurements at each timestamp
        """
        # Filter measurements with difference below threshold
        acceptable_measurements = pm25_instantaneous_df[pm25_instantaneous_df['pm25_diff'] <= threshold_diff]
        
        if len(acceptable_measurements) == 0:
            return 0, 0
        
        # Find maximum distance among acceptable measurements
        max_distance = acceptable_measurements['distance_km'].max()
        coverage_ratio = len(acceptable_measurements) / len(pm25_instantaneous_df)
        
        return max_distance, coverage_ratio

    # Test different thresholds
    thresholds = [3.0, 5.0, 7.5, 10.0, 15.0]
    results = []

    for threshold in thresholds:
        max_dist, coverage_ratio = find_coverage_threshold_instantaneous(pm25_instantaneous_df, threshold)
        results.append({
            'threshold_ugm3': threshold,
            'max_coverage_km': max_dist,
            'coverage_ratio': coverage_ratio,
            'n_acceptable_measurements': len(pm25_instantaneous_df[pm25_instantaneous_df['pm25_diff'] <= threshold])
        })

    threshold_results = pd.DataFrame(results)
    print("Coverage Analysis (Based on Instantaneous Measurements):")
    print(threshold_results)

    # Choose recommended threshold (default: 5 µg/m³)
    recommended_threshold = 5.0
    max_coverage_distance, coverage_ratio = find_coverage_threshold_instantaneous(pm25_instantaneous_df, recommended_threshold)

    print(f"\nRecommended maximum coverage distance: {max_coverage_distance:.2f} km")
    print(f"This covers {coverage_ratio:.1%} of instantaneous measurements within acceptable PM2.5 difference ({recommended_threshold} µg/m³)")
    
    # Additional analysis: percentage of pairs that meet threshold at least once
    pair_analysis = pm25_instantaneous_df.groupby(['sensor_1', 'sensor_2']).agg({
        'pm25_diff': ['min', 'mean', 'max'],
        'distance_km': 'first'
    }).reset_index()
    
    pair_analysis.columns = ['sensor_1', 'sensor_2', 'min_diff', 'mean_diff', 'max_diff', 'distance_km']
    
    pairs_within_threshold = pair_analysis[pair_analysis['min_diff'] <= recommended_threshold]
    print(f"\nPairs with at least one measurement within threshold: {len(pairs_within_threshold)}/{len(pair_analysis)} ({len(pairs_within_threshold)/len(pair_analysis):.1%})")

ERROR: No PM2.5 analysis data available!
Please run the previous cells first to generate the data.


## 6. Map Coverage Zones

In [51]:
import folium
import webbrowser

def create_coverage_map(sensor_locations, center_lat=None, center_lon=None):
    """
    Create interactive map with only sensor points (no coverage circles)
    """
    if center_lat is None:
        center_lat = sensor_locations['lat'].mean()
    if center_lon is None:
        center_lon = sensor_locations['lon'].mean()
    
    # Create base map
    m = folium.Map(location=[center_lat, center_lon], zoom_start=12)
    
    # Add only sensor markers
    for idx, sensor in sensor_locations.iterrows():
        folium.Marker(
            location=[sensor['lat'], sensor['lon']],
            popup=f"Sensor {sensor['sensor_id']}<br>Location: {sensor['location']}",
            icon=folium.Icon(color='red', icon='info-sign')
        ).add_to(m)
    
    return m

# Create and display map
coverage_map = create_coverage_map(sensor_locations)
coverage_map.save("coverage_map.html")
webbrowser.open("coverage_map.html")


True

## 7. Summary and Recommendations

In [41]:
# Generate comprehensive summary
summary = {
    'total_sensors': len(sensor_locations),
    'sensor_pairs_analyzed': len(pm25_analysis_df),
    'max_coverage_distance_km': max_coverage_distance,
    'pm25_threshold_ugm3': recommended_threshold,
    'coverage_ratio': coverage_ratio,
    'correlation_distance_pm25': corr_dist_diff
}

print("=" * 60)
print("SENSOR COVERAGE ANALYSIS SUMMARY")
print("=" * 60)
print(f"\n📊 Dataset Overview:")
print(f"   • Total PM2.5 sensors: {summary['total_sensors']}")
print(f"   • Sensor pairs analyzed: {summary['sensor_pairs_analyzed']}")

print(f"\n🎯 Coverage Distance Analysis:")
print(f"   • Maximum effective coverage distance: {summary['max_coverage_distance_km']:.2f} km")
print(f"   • Based on PM2.5 difference threshold: {summary['pm25_threshold_ugm3']} µg/m³")
print(f"   • Coverage ratio (pairs within threshold): {summary['coverage_ratio']:.1%}")
print(f"   • Correlation between distance and PM2.5 difference: {summary['correlation_distance_pm25']:.3f}")

print(f"\n💡 For Optimization:")
print(f"   • Use {summary['max_coverage_distance_km']:.2f} km as coverage radius in objective function")
print(f"   • Weight sensors by their unique coverage area")
print(f"   • Minimize overlap while maximizing total coverage")

print("=" * 60)

SENSOR COVERAGE ANALYSIS SUMMARY

📊 Dataset Overview:
   • Total PM2.5 sensors: 43
   • Sensor pairs analyzed: 0

🎯 Coverage Distance Analysis:
   • Maximum effective coverage distance: 17.90 km
   • Based on PM2.5 difference threshold: 5.0 µg/m³
   • Coverage ratio (pairs within threshold): 100.0%
   • Correlation between distance and PM2.5 difference: nan

💡 For Optimization:
   • Use 17.90 km as coverage radius in objective function
   • Weight sensors by their unique coverage area
   • Minimize overlap while maximizing total coverage


## 8. Export Results

In [43]:
# Save key results to CSV files
sensor_pairs_df.to_csv('sensor_distances.csv', index=False)
pm25_analysis_df.to_csv('pm25_distance_analysis.csv', index=False)
threshold_results.to_csv('coverage_thresholds.csv', index=False)

print("Results exported:")
print("• sensor_distances.csv - Pairwise distances between sensors")
print("• pm25_distance_analysis.csv - PM2.5 differences vs distance")
print("• coverage_thresholds.csv - Coverage analysis for different thresholds")

Results exported:
• sensor_distances.csv - Pairwise distances between sensors
• pm25_distance_analysis.csv - PM2.5 differences vs distance
• coverage_thresholds.csv - Coverage analysis for different thresholds
